<a href="https://colab.research.google.com/github/XQSUNgithub/TRELLIS.2/blob/main/add-example-notebook/example.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TRELLIS.2 Inference on Google Colab
This notebook sets up the environment to run Microsoft's 4-billion parameter TRELLIS.2 model on a standard Google Colab (Tested using L4 GPU)

**Prerequisites:**
Before running this notebook, you must have a Hugging Face account and accept the usage agreements for the following gated models:
1. Background Removal: [briaai/RMBG-2.0](https://huggingface.co/briaai/RMBG-2.0)
2. Image Conditioning: [Facebook DINOv3](https://huggingface.co/facebook/dinov3-vitl16-pretrain-lvd1689m)

You will also need to add your Hugging Face Read Token to Colab's **Secrets** tab (the key icon on the left sidebar) and name it `HF_TOKEN`.

In [ ]:
# Clone the repository and install all custom CUDA extensions
!git clone -b main https://github.com/microsoft/TRELLIS.2.git --recursive

%cd /content/TRELLIS.2/

# Run the setup script to compile cumesh, o-voxel, and flexgemm
!. ./setup.sh --new-env --basic --flash-attn --nvdiffrast --nvdiffrec --cumesh --o-voxel --flexgemm

# Downgrade libraries to avoid meta tensor and internal utility conflicts
!pip install "Pillow<10.0.0"
!pip install "transformers<5.0.0"

### 🛑 STOP AND RESTART RUNTIME
Because we installed specific versions of `Pillow` and `transformers`, Python will crash if you do not clear its active memory.

Go to the top menu: **Runtime > Restart session** (or Restart runtime).
Do **not** run the setup cell above again. Proceed directly to the cell below.

In [ ]:
# Re-enter the directory after the restart
%cd /content/TRELLIS.2/

from huggingface_hub import login
from google.colab import userdata

# Authenticate with Hugging Face to download the gated models
print("Logging into Hugging Face...")
hf_token = userdata.get('HF_TOKEN')
login(hf_token)

In [ ]:
import os
os.environ['OPENCV_IO_ENABLE_OPENEXR'] = '1'
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"  # Can save GPU memory
import cv2
import imageio
from PIL import Image
import torch
from trellis2.pipelines import Trellis2ImageTo3DPipeline
from trellis2.utils import render_utils
from trellis2.renderers import EnvMap
import o_voxel

# 1. Setup Environment Map
envmap = EnvMap(torch.tensor(
    cv2.cvtColor(cv2.imread('assets/hdri/forest.exr', cv2.IMREAD_UNCHANGED), cv2.COLOR_BGR2RGB),
    dtype=torch.float32, device='cuda'
))

# 2. Load Pipeline
pipeline = Trellis2ImageTo3DPipeline.from_pretrained("microsoft/TRELLIS.2-4B")
pipeline.cuda()

# 3. Load Image & Run
image = Image.open("assets/example_image/T.png")
mesh = pipeline.run(image)[0]
mesh.simplify(16777216) # nvdiffrast limit

# 4. Render Video
video = render_utils.make_pbr_vis_frames(render_utils.render_video(mesh, envmap=envmap))
imageio.mimsave("sample.mp4", video, fps=15)

# 5. Export to GLB
glb = o_voxel.postprocess.to_glb(
    vertices            =   mesh.vertices,
    faces               =   mesh.faces,
    attr_volume         =   mesh.attrs,
    coords              =   mesh.coords,
    attr_layout         =   mesh.layout,
    voxel_size          =   mesh.voxel_size,
    aabb                =   [[-0.5, -0.5, -0.5], [0.5, 0.5, 0.5]],
    decimation_target   =   1000000,
    texture_size        =   4096,
    remesh              =   True,
    remesh_band         =   1,
    remesh_project      =   0,
    verbose             =   True
)
glb.export("sample.glb", extension_webp=True)

In [20]:
!python data_toolkit/build_metadata.py ObjaverseXL --source sketchfab --root datasets/ObjaverseXL_sketchfab

Statistics:
  - Number of assets: 168307
  - Number of assets downloaded: 0



In [21]:
!python data_toolkit/download.py ObjaverseXL --root datasets/ObjaverseXL_sketchfab --world_size 160000

Processing 1 objects...
2026-03-17 13:27:26.465 | INFO     | objaverse.xl.sketchfab:download_objects:508 - Found 0 objects already downloaded
2026-03-17 13:27:26.465 | INFO     | objaverse.xl.sketchfab:download_objects:529 - Downloading 1 new objects across 26 processes
100% 1/1 [00:01<00:00,  1.21s/it]


18e8e405446849afb22b3760d3c73a31

In [22]:
!mv /content/TRELLIS.2/sample.glb /content/TRELLIS.2/datasets/ObjaverseXL_sketchfab/raw/hf-objaverse-v1/glbs/000-069

In [23]:
!python data_toolkit/build_metadata.py ObjaverseXL --root datasets/ObjaverseXL_sketchfab

Loading previous metadata...
Statistics:
  - Number of assets: 168307
  - Number of assets downloaded: 1



In [24]:
!python data_toolkit/dump_mesh.py ObjaverseXL --root datasets/ObjaverseXL_sketchfab
!python data_toolkit/dump_pbr.py ObjaverseXL --root datasets/ObjaverseXL_sketchfab
!python data_toolkit/asset_stats.py --root datasets/ObjaverseXL_sketchfab

Checking blender...
Found 0 dumped mesh
Processing 1 objects...
Dumping mesh: 100% 1/1 [00:05<00:00,  5.38s/it]
Checking blender...
Blender 4.5.1 LTS (hash b0a72b245dcf built 2025-07-29 06:24:35)
Looking in links: /tmp/tmpa6cc6y7t

[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: /tmp/blender-4.5.1-linux-x64/4.5/python/bin/python3.11 -m pip install --upgrade pip

Blender quit
Found 0 dumped PBRs
Processing 1 objects...
Dumping PBR: 100% 1/1 [00:50<00:00, 50.26s/it]
Processing 1 objects...
Processing objects: 100% 1/1 [00:00<00:00, 15.91it/s]


In [25]:
!python data_toolkit/build_metadata.py ObjaverseXL --root datasets/ObjaverseXL_sketchfab

Loading previous metadata...
Statistics:
  - Number of assets: 168307
  - Number of assets downloaded: 1
  - Number of assets with mesh dumped: True
  - Number of assets with PBR dumped: True
  - Number of assets with asset stats: 1



In [26]:
!python data_toolkit/dual_grid.py ObjaverseXL --root datasets/ObjaverseXL_sketchfab --resolution 256,512,1024
!python data_toolkit/voxelize_pbr.py ObjaverseXL --root datasets/ObjaverseXL_sketchfab --resolution 256,512,1024

Processing 1 objects...
Dual griding: 100% 1/1 [00:54<00:00, 54.44s/it]
Processing 1 objects...
Voxelizing: 100% 1/1 [00:57<00:00, 57.21s/it]


In [27]:
!python data_toolkit/encode_shape_latent.py --root datasets/ObjaverseXL_sketchfab --resolution 512
!python data_toolkit/encode_pbr_latent.py --root datasets/ObjaverseXL_sketchfab --resolution 512
!python data_toolkit/encode_shape_latent.py --root datasets/ObjaverseXL_sketchfab --resolution 1024
!python data_toolkit/encode_pbr_latent.py --root datasets/ObjaverseXL_sketchfab --resolution 1024

# Update metadata
!python data_toolkit/build_metadata.py ObjaverseXL --root datasets/ObjaverseXL_sketchfab

# Encode SS Latents
!python data_toolkit/encode_ss_latent.py --root datasets/ObjaverseXL_sketchfab --shape_latent_name shape_enc_next_dc_f16c32_fp16_1024 --resolution 64

# Final Metadata Update
!python data_toolkit/build_metadata.py ObjaverseXL --root datasets/ObjaverseXL_sketchfab

[SPARSE] Conv backend: flex_gemm; Attention backend: flash_attn
Index(['sha256', 'file_identifier', 'aesthetic_score', 'captions',
       'dual_grid_converted'],
      dtype='object')
Filtering existing objects: 100% 1/1 [00:00<00:00, 2219.21it/s]
Found 0 processed objects
Processing 1 objects...
Extracting latents: 100% 1/1 [00:00<00:00,  1.13it/s]
[SPARSE] Conv backend: flex_gemm; Attention backend: flash_attn
Filtering existing objects: 100% 1/1 [00:00<00:00, 2007.80it/s]
Found 0 processed objects
Processing 1 objects...
Extracting latents: 100% 1/1 [00:00<00:00,  1.10it/s]
[SPARSE] Conv backend: flex_gemm; Attention backend: flash_attn
Index(['sha256', 'file_identifier', 'aesthetic_score', 'captions',
       'dual_grid_converted'],
      dtype='object')
Filtering existing objects: 100% 1/1 [00:00<00:00, 2148.72it/s]
Found 0 processed objects
Processing 1 objects...
Extracting latents: 100% 1/1 [00:06<00:00,  6.07s/it]
[SPARSE] Conv backend: flex_gemm; Attention backend: flash_attn


In [36]:
!python data_toolkit/render_cond.py ObjaverseXL --root datasets/ObjaverseXL_sketchfab

Checking blender...
Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:4 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:6 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:7 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:8 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:9 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Hit:11 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry missp

In [ ]:
!zip -r renders_cond.zip /content/TRELLIS.2/datasets/ObjaverseXL_sketchfab/renders_cond

In [33]:
from google.colab import files

files.download('renders_cond.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>